In [1]:
import json
from pathlib import Path
import pandas as pd

raw_dir = Path("../data/raw")
cleaned_dir = Path("../data/cleaned")
cleaned_dir.mkdir(parents=True, exist_ok=True)

with open(raw_dir / "products_raw.json", "r", encoding="utf-8") as f:
    products_df = pd.DataFrame(json.load(f)["products"])

with open(raw_dir / "users_raw.json", "r", encoding="utf-8") as f:
    users_df = pd.DataFrame(json.load(f)["users"])

with open(raw_dir / "carts_raw.json", "r", encoding="utf-8") as f:
    carts_df = pd.DataFrame(json.load(f)["carts"])

print("Shapes:", products_df.shape, users_df.shape, carts_df.shape)
products_df.head(3)

Shapes: (100, 22) (100, 28) (100, 7)


,id,title,description,category,price,discountPercentage,rating,stock,tags,brand,...,dimensions,warrantyInformation,shippingInformation,availabilityStatus,reviews,returnPolicy,minimumOrderQuantity,meta,images,thumbnail
0,1,Essence Mascara Lash Princess,The Essence Mascara Lash Princess is a popular...,beauty,9.99,10.48,2.56,99,"[beauty, mascara]",Essence,...,"{'width': 15.14, 'height': 13.08, 'depth': 22.99}",1 week warranty,Ships in 3-5 business days,In Stock,"[{'rating': 3, 'comment': 'Would not recommend...",No return policy,48,"{'createdAt': '2025-10-09T14:47:01.588Z', 'upd...",[https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...
1,2,Eyeshadow Palette with Mirror,The Eyeshadow Palette with Mirror offers a ver...,beauty,19.99,18.19,2.86,34,"[beauty, eyeshadow]",Glamour Beauty,...,"{'width': 9.26, 'height': 22.47, 'depth': 27.67}",1 year warranty,Ships in 2 weeks,In Stock,"[{'rating': 5, 'comment': 'Great product!', 'd...",7 days return policy,20,"{'createdAt': '2024-12-03T11:29:45.706Z', 'upd...",[https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...
2,3,Powder Canister,The Powder Canister is a finely milled setting...,beauty,14.99,9.84,4.64,89,"[beauty, face powder]",Velvet Touch,...,"{'width': 29.27, 'height': 27.93, 'depth': 20.59}",3 months warranty,Ships in 1-2 business days,In Stock,"[{'rating': 4, 'comment': 'Would buy again!', ...",No return policy,22,"{'createdAt': '2025-01-08T18:30:12.844Z', 'upd...",[https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...


In [2]:
print("Duplicate IDs -> products:", products_df["id"].duplicated().sum(),
      "| users:", users_df["id"].duplicated().sum(),
      "| carts:", carts_df["id"].duplicated().sum())

print("\nMissing values in products:\n", products_df.isna().sum()[products_df.isna().sum() > 0])
print("\nMissing values in users:\n", users_df.isna().sum()[users_df.isna().sum() > 0])

# Step 1: Drop duplicates by primary id
products_df = products_df.drop_duplicates(subset=["id"]).copy()
users_df = users_df.drop_duplicates(subset=["id"]).copy()
carts_df = carts_df.drop_duplicates(subset=["id"]).copy()

# Step 2: Fill missing values
products_df["brand"] = products_df["brand"].fillna("Unbranded")
users_df["maidenName"] = users_df["maidenName"].fillna("")

Duplicate IDs -> products: 0 | users: 0 | carts: 0

Missing values in products:
 brand    62
dtype: int64

Missing values in users:
 Series([], dtype: int64)


In [3]:
# Step 3: Standardize column names to snake_case
products_df = products_df.rename(columns={
    "discountPercentage": "discount_percentage",
    "warrantyInformation": "warranty_info",
    "shippingInformation": "shipping_info",
    "availabilityStatus": "availability_status",
    "returnPolicy": "return_policy",
    "minimumOrderQuantity": "min_order_qty",
})

users_df = users_df.rename(columns={
    "firstName": "first_name",
    "lastName": "last_name",
    "maidenName": "maiden_name",
    "birthDate": "birth_date",
    "bloodGroup": "blood_group",
    "eyeColor": "eye_color",
})

carts_df = carts_df.rename(columns={
    "userId": "user_id",
    "discountedTotal": "discounted_total",
    "totalProducts": "total_products",
    "totalQuantity": "total_quantity",
})

products_df[["id", "title", "discount_percentage", "availability_status"]].head(3)

,id,title,discount_percentage,availability_status
0,1,Essence Mascara Lash Princess,10.48,In Stock
1,2,Eyeshadow Palette with Mirror,18.19,In Stock
2,3,Powder Canister,9.84,In Stock


In [4]:
# Step 4: Extract nested dictionary/list fields before dropping nested columns
users_df["city"] = users_df["address"].apply(lambda a: a.get("city") if isinstance(a, dict) else None)
users_df["country"] = users_df["address"].apply(lambda a: a.get("country") if isinstance(a, dict) else None)
products_df["created_at_raw"] = products_df["meta"].apply(lambda m: m.get("createdAt") if isinstance(m, dict) else None)
products_df["tag_count"] = products_df["tags"].apply(lambda t: len(t) if isinstance(t, list) else 0)

# Step 5: Split text into clean columns
users_df["full_name"] = (
    users_df["first_name"].astype(str).str.strip() + " " + users_df["last_name"].astype(str).str.strip()
).str.strip()
products_df["shipping_summary"] = (
    products_df["shipping_info"].astype(str).str.split("-").str[0].str.strip()
)

users_df[["full_name", "city", "country"]].head(5)

,full_name,city,country
0,Emily Johnson,Phoenix,United States
1,Michael Williams,Houston,United States
2,Sophia Brown,Washington,United States
3,James Davis,Seattle,United States
4,Emma Miller,Jacksonville,United States


In [5]:
# Step 6: Standardize text strings
products_df["category"] = (
    products_df["category"].astype(str).str.strip().str.lower().str.replace("-", " ", regex=False)
)
products_df["brand"] = products_df["brand"].astype(str).str.strip()
products_df["availability_status"] = products_df["availability_status"].astype(str).str.strip().str.title()

users_df["gender"] = users_df["gender"].astype(str).str.strip().str.lower()
users_df["email"] = users_df["email"].astype(str).str.strip().str.lower()

products_df["category"].value_counts().head(10)

category
kitchen accessories    30
groceries              27
mens watches            6
beauty                  5
fragrances              5
furniture               5
home decoration         5
laptops                 5
mens shirts             5
mens shoes              5
Name: count, dtype: int64

In [6]:
# Step 7: Convert numeric and integer columns
for col in ["price", "discount_percentage", "rating", "weight"]:
    products_df[col] = pd.to_numeric(products_df[col], errors="coerce")

for col in ["stock", "min_order_qty", "tag_count"]:
    products_df[col] = pd.to_numeric(products_df[col], errors="coerce").fillna(0).astype(int)

for col in ["age", "height", "weight"]:
    users_df[col] = pd.to_numeric(users_df[col], errors="coerce")

for col in ["total", "discounted_total"]:
    carts_df[col] = pd.to_numeric(carts_df[col], errors="coerce")

for col in ["total_products", "total_quantity"]:
    carts_df[col] = pd.to_numeric(carts_df[col], errors="coerce").fillna(0).astype(int)

# Step 8: Parse dates
users_df["birth_date"] = pd.to_datetime(users_df["birth_date"], errors="coerce")
products_df["created_at"] = pd.to_datetime(products_df["created_at_raw"], errors="coerce")

products_df[["price", "discount_percentage", "stock", "created_at"]].dtypes

price                              float64
discount_percentage                float64
stock                                int64
created_at             datetime64[us, UTC]
dtype: object

In [7]:
# Step 9: Derived metrics
products_df["discounted_price"] = (
    products_df["price"] * (1 - products_df["discount_percentage"] / 100)
).round(2)
products_df["stock_value"] = (products_df["discounted_price"] * products_df["stock"]).round(2)

carts_df["cart_savings"] = (carts_df["total"] - carts_df["discounted_total"]).round(2)
carts_df["avg_item_value"] = (
    carts_df["discounted_total"] / carts_df["total_quantity"].replace(0, pd.NA)
).round(2)

# Unpack nested products inside each cart into a clean line-items table
cart_items_rows = []
for _, row in carts_df.iterrows():
    items = row.get("products")
    if isinstance(items, list):
        for item in items:
            if isinstance(item, dict):
                cart_items_rows.append({
                    "cart_id": row["id"],
                    "user_id": row["user_id"],
                    "product_id": item.get("id"),
                    "title": item.get("title"),
                    "price": item.get("price"),
                    "quantity": item.get("quantity"),
                    "total": item.get("total"),
                    "discount_percentage": item.get("discountPercentage"),
                    "discounted_total": item.get("discountedTotal"),
                })

cart_items_df = pd.DataFrame(cart_items_rows).drop_duplicates().copy()
for col in ["price", "total", "discount_percentage", "discounted_total"]:
    cart_items_df[col] = pd.to_numeric(cart_items_df[col], errors="coerce")
cart_items_df["quantity"] = pd.to_numeric(cart_items_df["quantity"], errors="coerce").fillna(0).astype(int)

products_df[["title", "price", "discounted_price", "stock", "stock_value"]].head(5)

,title,price,discounted_price,stock,stock_value
0,Essence Mascara Lash Princess,9.99,8.94,99,885.06
1,Eyeshadow Palette with Mirror,19.99,16.35,34,555.90
2,Powder Canister,14.99,13.51,89,1202.39
3,Red Lipstick,12.99,11.41,91,1038.31
4,Red Nail Polish,8.99,7.96,79,628.84


In [8]:
# Step 10: Drop noisy, raw nested, or unused columns
product_drops = ["thumbnail", "images", "meta", "created_at_raw", "reviews", "dimensions", "tags", "shipping_info"]
user_drops = ["password", "ssn", "ein", "ip", "macAddress", "userAgent", "address", "hair", "bank", "company", "crypto", "image"]
cart_drops = ["products"]

products_clean = products_df.drop(columns=[c for c in product_drops if c in products_df.columns])
users_clean = users_df.drop(columns=[c for c in user_drops if c in users_df.columns])
carts_clean = carts_df.drop(columns=[c for c in cart_drops if c in carts_df.columns])

# Step 11: Export cleaned CSVs
products_clean.to_csv(cleaned_dir / "products_cleaned.csv", index=False)
users_clean.to_csv(cleaned_dir / "users_cleaned.csv", index=False)
carts_clean.to_csv(cleaned_dir / "carts_cleaned.csv", index=False)
cart_items_df.to_csv(cleaned_dir / "cart_items_cleaned.csv", index=False)

print("Saved all 4 cleaned CSVs successfully!")
print("Final shapes:", products_clean.shape, users_clean.shape, carts_clean.shape, cart_items_df.shape)

Saved all 4 cleaned CSVs successfully!
Final shapes: (100, 20) (100, 19) (100, 8) (376, 9)


In [9]:
import re

cleaned_tables = {
    "products": products_df,
    "users": users_df,
    "carts": carts_df,
    "cart_items": cart_items_df,
}

# 1. Collapse internal multiple spaces on object/string columns
for name, df in cleaned_tables.items():
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].apply(
            lambda v: re.sub(r"\s+", " ", v).strip() if isinstance(v, str) else v
        )

# 2. Audit remaining missing values / NaTs across all 4 tables
for name, df in cleaned_tables.items():
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    print(f"--- {name} missing/NaT counts ---")
    print(nulls if not nulls.empty else "None")

# 3. Quick numeric & date sanity checks
print("\nAny NaT in parsed dates?")
print("users.birth_date NaT:", users_df["birth_date"].isna().sum())
print("products.created_at NaT:", products_df["created_at"].isna().sum())

print("\nNegative values check:")
print("products price < 0:", (products_df["price"] < 0).sum(), "| stock < 0:", (products_df["stock"] < 0).sum())
print("cart_items qty <= 0:", (cart_items_df["quantity"] <= 0).sum(), "| total < 0:", (cart_items_df["total"] < 0).sum())

--- products missing/NaT counts ---
None
--- users missing/NaT counts ---
None
--- carts missing/NaT counts ---
None
--- cart_items missing/NaT counts ---
None

Any NaT in parsed dates?
users.birth_date NaT: 0
products.created_at NaT: 0

Negative values check:
products price < 0: 0 | stock < 0: 0
cart_items qty <= 0: 0 | total < 0: 0
